In [1]:
import sys

sys.path.insert(0, "..")
from runner.utils import (
    allocate_benchmarks,
    create_benchmark_campaign,
    display_speedups,
    load_benchmark_metadata,
    load_results,
)

In [2]:
# If a util function was modified, use this cell to reload it without having to restart the kernel
%run ../runner/utils.py

## Create benchmark campaign(s)

### 20260303 Run spatial resolutions 2, 4

In [3]:
benchmarks_df = load_benchmark_metadata()
benchs_to_run = benchmarks_df[
    (benchmarks_df["Benchmark"] == "pypsa-de-elec")
    & (benchmarks_df["Instance"].str.endswith("-1h"))
]
# NOTE: picking only even num of nodes >= 2 to save costs
benchs_to_run = benchs_to_run[
    benchs_to_run["Instance"].map(
        lambda i: (lambda n: n == 2 and n % 2 == 0)(int(i.split("-")[0]))
    )
]
len(benchs_to_run)

1

In [4]:
# Create campaign: 1 instance per VM, latest solvers only (because I marked all as Ls in the metadata)

benchs_to_run.loc[:, "Num. variables"] = 1  # Dummy value to run 1 instance per VM
vm_yamls = allocate_benchmarks(
    benchs_to_run,
    "Num. variables",
    len(benchs_to_run),
    machine_type="c4-standard-8",  # NOTE: picked a smaller machine size to save costs!
    timeout_seconds=24 * 60 * 60,
    solvers="cbc gurobi highs-hipo highs-ipm highs",  # NOTE: skipped SCIP since it's not built for LPs
    years=[2024, 2025],  # latest solvers only, so skip creating older conda envs
)

# # Use a different zone for some instances to avoid exceeding CPU quota
# for y in vm_yamls:
#     if int(y['benchmarks']['pypsa-de-elec']['Sizes'][0]['Name'].split('-')[0]) >= 20:
#         print(y['benchmarks']['pypsa-de-elec']['Sizes'][0]['Name'])
#         y['zone'] = 'us-east4-a'

create_benchmark_campaign(
    "20260303-all-pypsa-de-small-sizes",
    "all-pypsa-de-small-sizes",
    vm_yamls,
)

Allocated. Estimated runtime: 0.0h
  VM 00: 1 instances, 0.0h
Created directory and files in ../infrastructure/benchmarks/20260303-all-pypsa-de-small-sizes
Run this campaign from the infrastructure/ directory using the command:
tofu apply -var-file benchmarks/20260303-all-pypsa-de-small-sizes/run.tfvars -state=states/20260303-all-pypsa-de-small-sizes.tfstate


### 20260223 Run spatial resolutions 2, 4, .., 20

In [ ]:
benchmarks_df = load_benchmark_metadata()
benchs_to_run = benchmarks_df[
    (benchmarks_df["Benchmark"] == "pypsa-de-elec-uc")
    & (benchmarks_df["Instance"].str.endswith("-1h"))
]
# NOTE: picking only even num of nodes >= 2 to save costs
benchs_to_run = benchs_to_run[
    benchs_to_run["Instance"].map(
        lambda i: (lambda n: n >= 2 and n % 2 == 0)(int(i.split("-")[0]))
    )
]
len(benchs_to_run)

In [ ]:
# Create campaign: 1 instance per VM, latest solvers only (because I marked all as Ls in the metadata)

benchs_to_run.loc[:, "Num. variables"] = 1  # Dummy value to run 1 instance per VM
vm_yamls = allocate_benchmarks(
    benchs_to_run,
    "Num. variables",
    len(benchs_to_run),
    machine_type="c4-standard-8",  # NOTE: picked a smaller machine size to save costs!
    timeout_seconds=24 * 60 * 60,
    solvers="cbc gurobi highs-hipo highs-ipm highs",  # NOTE: skipped SCIP since it's not built for LPs
    years=[2024, 2025],  # latest solvers only, so skip creating older conda envs
)

# # Use a different zone for some instances to avoid exceeding CPU quota
# for y in vm_yamls:
#     if int(y['benchmarks']['pypsa-de-elec']['Sizes'][0]['Name'].split('-')[0]) >= 20:
#         print(y['benchmarks']['pypsa-de-elec']['Sizes'][0]['Name'])
#         y['zone'] = 'us-east4-a'

create_benchmark_campaign(
    "20260211-all-pypsa-de-uc-sizes",
    "all-pypsa-de-uc-sizes",
    vm_yamls,
)

## Monitor runs

To view running VMs:
```
gcloud compute instances list | sort | tee /dev/tty | grep benchmark-instance | grep -i running | wc -l
```

To SSH into a running VM and see what's happening:
```
gcloud compute ssh projects/compute-app-427709/zones/us-central1-a/instances/benchmark-instance-more-pypsa-de-sizes-04
tail -f /var/log/startup-script.log
cat /solver-benchmark/results/benchmark_results.csv
```

## Inspect results

To download results:
```
gsutil -m rsync -r gs://solver-benchmarks/results ./results/gcp-results/
```

In [ ]:
results, variability = load_results(
    ["../results/gcp-results/20260211-all-pypsa-de-sizes/"]
)
benchmarks_df = load_benchmark_metadata()
display_speedups(results, benchmarks_df)